# Breast Cancer Detection Using Deep Learning

This notebook trains and compares multiple convolutional neural network (CNN) backbones (transfer learning) for **benign vs malignant** classification on mammography images.

## Contents
1. Setup
2. Data configuration
3. Input pipeline
4. Model factory
5. Training loop
6. Evaluation and comparison


## 1) Setup

In [ ]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


## 2) Data configuration

### Dataset link
CBIS-DDSM (The Cancer Imaging Archive):
https://wiki.cancerimagingarchive.net/display/Public/CBIS-DDSM

### Expected folder format
This notebook expects a simple folder structure:

```
DATA_ROOT/
  train/
    benign/
    malignant/
  val/
    benign/
    malignant/
  test/
    benign/
    malignant/
```

Update `DATA_ROOT` below to match your local setup.

In [ ]:
# Update this path
DATA_ROOT = r"/path/to/your/cbis-ddsm-splits"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_dir = os.path.join(DATA_ROOT, "train")
val_dir   = os.path.join(DATA_ROOT, "val")
test_dir  = os.path.join(DATA_ROOT, "test")

for d in [train_dir, val_dir, test_dir]:
    print(d, 'exists:', os.path.isdir(d))


## 3) Input pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='binary',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    labels='inferred',
    label_mode='binary',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    labels='inferred',
    label_mode='binary',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

class_names = train_ds.class_names
print('Classes:', class_names)

def configure(ds):
    return ds.cache().prefetch(buffer_size=AUTOTUNE)

train_ds = configure(train_ds)
val_ds   = configure(val_ds)
test_ds  = configure(test_ds)


### Sample visualization

In [ ]:
plt.figure(figsize=(10, 8))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        lbl = int(labels[i].numpy()[0])
        plt.title(class_names[lbl])
        plt.axis('off')
plt.tight_layout()
plt.show()


## 4) Model factory (transfer learning)

The goal is a fair comparison across backbones by keeping the pipeline consistent.

In [ ]:
DATA_AUG = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.03),
    layers.RandomZoom(0.05),
], name='augmentation')

BACKBONES = {
    'VGG16': keras.applications.VGG16,
    'VGG19': keras.applications.VGG19,
    'ResNet50': keras.applications.ResNet50,
    'ResNet101': keras.applications.ResNet101,
    'InceptionV3': keras.applications.InceptionV3,
    'InceptionResNetV2': keras.applications.InceptionResNetV2,
    'MobileNet': keras.applications.MobileNet,
    'DenseNet121': keras.applications.DenseNet121,
}

def get_preprocess_fn(backbone_name: str):
    """Return the correct preprocess_input for a given backbone."""
    if backbone_name in ('VGG16', 'VGG19'):
        return keras.applications.vgg16.preprocess_input
    if backbone_name == 'ResNet50':
        return keras.applications.resnet50.preprocess_input
    if backbone_name == 'ResNet101':
        return keras.applications.resnet.preprocess_input
    if backbone_name == 'InceptionV3':
        return keras.applications.inception_v3.preprocess_input
    if backbone_name == 'InceptionResNetV2':
        return keras.applications.inception_resnet_v2.preprocess_input
    if backbone_name == 'MobileNet':
        return keras.applications.mobilenet.preprocess_input
    if backbone_name == 'DenseNet121':
        return keras.applications.densenet.preprocess_input
    raise ValueError(f'Unknown backbone: {backbone_name}')

def build_model(backbone_name: str, lr: float = 1e-4, dropout: float = 0.3):
    Backbone = BACKBONES[backbone_name]
    base = Backbone(
        include_top=False,
        weights='imagenet',
        input_shape=IMG_SIZE + (3,),
    )
    base.trainable = False

    preprocess = get_preprocess_fn(backbone_name)

    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = DATA_AUG(inputs)
    x = preprocess(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs, name=f'{backbone_name}_binary')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=[keras.metrics.BinaryAccuracy(name='accuracy'), keras.metrics.AUC(name='auc')]
    )
    return model


## 5) Training utilities

In [ ]:
OUT_DIR = Path('results')
OUT_DIR.mkdir(exist_ok=True)

def callbacks_for(name: str):
    ckpt_path = OUT_DIR / f'{name}.keras'
    return [
        keras.callbacks.ModelCheckpoint(
            filepath=str(ckpt_path),
            monitor='val_auc',
            mode='max',
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor='val_auc',
            mode='max',
            patience=4,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_auc',
            mode='max',
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1,
        )
    ]

def plot_curves(history, name='model'):
    hist = pd.DataFrame(history.history)

    plt.figure(figsize=(7, 4))
    if 'accuracy' in hist and 'val_accuracy' in hist:
        plt.plot(hist['accuracy'], label='train')
        plt.plot(hist['val_accuracy'], label='val')
        plt.title(f'{name} | Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.tight_layout()
        plt.show()

    plt.figure(figsize=(7, 4))
    if 'auc' in hist and 'val_auc' in hist:
        plt.plot(hist['auc'], label='train')
        plt.plot(hist['val_auc'], label='val')
        plt.title(f'{name} | AUC')
        plt.xlabel('Epoch')
        plt.ylabel('AUC')
        plt.legend()
        plt.tight_layout()
        plt.show()


## 6) Evaluation helpers

In [ ]:
def get_labels_and_probs(model, ds):
    y_true, y_prob = [], []
    for x, y in ds:
        p = model.predict(x, verbose=0).reshape(-1)
        y_prob.extend(p.tolist())
        y_true.extend(y.numpy().reshape(-1).tolist())
    return np.array(y_true).astype(int), np.array(y_prob)

def evaluate_binary(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else float('nan'),
    }
    return metrics

def plot_roc(y_true, y_prob, title='ROC'):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f'AUC={auc:.3f}')
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.title(title)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

def show_confusion_and_report(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    print('Confusion matrix (rows=true, cols=pred):')
    print(cm)
    print('\nClassification report:')
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))


## 7) Train and compare backbones

Edit `MODELS_TO_RUN` to control which architectures you train in this run.

In [ ]:
MODELS_TO_RUN = [
    'DenseNet121',
    'MobileNet',
    'ResNet50',
]

EPOCHS = 12
LR = 1e-4
THRESHOLD = 0.5

results = []

for name in MODELS_TO_RUN:
    print('\n' + '='*70)
    print(f'Model: {name}')
    print('='*70)

    model = build_model(name, lr=LR)
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks_for(name),
        verbose=1,
    )
    plot_curves(history, name=name)

    y_true, y_prob = get_labels_and_probs(model, test_ds)
    metrics = evaluate_binary(y_true, y_prob, threshold=THRESHOLD)
    metrics['model'] = name
    results.append(metrics)

    print('Test metrics:', metrics)
    plot_roc(y_true, y_prob, title=f'{name} | ROC')
    show_confusion_and_report(y_true, y_prob, threshold=THRESHOLD)


## 8) Comparison table

This table summarizes the test metrics for the selected models.

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df[['model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']].sort_values('roc_auc', ascending=False)
results_df


In [ ]:
out_csv = OUT_DIR / 'metrics_summary.csv'
results_df.to_csv(out_csv, index=False)
print('Saved:', out_csv)
